This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [1]:
import great_expectations
context = great_expectations.get_context()
import logging

In [2]:
import yaml

In [3]:
from datetime import date,datetime

In [4]:
logging.basicConfig(level=logging.DEBUG, force = True)

In [5]:
connection_string = """bigquery://world-fishing-827/tech_great_expectations_temp_ttl_7d?\
credentials_path=/mnt/encrypted_data/git/api_keys/world-fishing-827-02584bdf5326.json"""

In [6]:
with open("datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [7]:
datasource_config.get("project")

'gfw-google-827'

In [8]:
gx_project = datasource_config.get("project")
gx_datasource = context.get_datasource(gx_project)

DEBUG:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.__fields_set__ assets added
DEBUG:great_expectations.datasource.fluent.fluent_base_model:QueryAsset.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SplitterColumnValue.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:QueryAsset.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SplitterColumnValue.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:QueryAsset.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SplitterColumnValue.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:QueryAsset.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SplitterColumnValue.__fields_set__ assets discarded
DEBUG:great_expectations.data

In [9]:
gx_datasource.get_asset_names()

{'messages-2.5',
 'messages-3.0.0',
 'satellite_timing_offsets-2.5',
 'satellite_timing_offsets-3.0.0',
 'segment_info-2.5',
 'segment_info-3.0.0',
 'segment_vessel-2.5',
 'segment_vessel-3.0.0',
 'segs_activity-2.5',
 'segs_activity-3.0.0',
 'segs_activity_daily-2.5',
 'segs_activity_daily-3.0.0',
 'ssvids_identities-2.5',
 'ssvids_identities-3.0.0',
 'ssvids_identities_daily-2.5',
 'ssvids_identities_daily-3.0.0',
 'stats_daily-2.5',
 'stats_daily-3.0.0',
 'vessel_info-2.5',
 'vessel_info-3.0.0'}

In [10]:
context.list_expectation_suite_names()

['gfw-google-827.alerts.messages.2-5',
 'gfw-google-827.alerts.messages.3-0-0',
 'gfw-google-827.alerts.messages_positions.2-5',
 'gfw-google-827.alerts.messages_positions.3-0-0',
 'gfw-google-827.alerts.satellite_timing_offsets.2-5',
 'gfw-google-827.alerts.satellite_timing_offsets.3-0-0',
 'gfw-google-827.alerts.segment_info.2-5',
 'gfw-google-827.alerts.segment_info.3-0-0',
 'gfw-google-827.alerts.segment_vessel.2-5',
 'gfw-google-827.alerts.segment_vessel.3-0-0',
 'gfw-google-827.alerts.segs_activity.2-5',
 'gfw-google-827.alerts.segs_activity.3-0-0',
 'gfw-google-827.alerts.segs_activity_daily.2-5',
 'gfw-google-827.alerts.segs_activity_daily.3-0-0',
 'gfw-google-827.alerts.ssvids_identities.2-5',
 'gfw-google-827.alerts.ssvids_identities.3-0-0',
 'gfw-google-827.alerts.ssvids_identities_daily.2-5',
 'gfw-google-827.alerts.ssvids_identities_daily.3-0-0',
 'gfw-google-827.alerts.stats_daily.2-5',
 'gfw-google-827.alerts.stats_daily.3-0-0',
 'gfw-google-827.alerts.vessel_info.2-5',


In [19]:
for current_expectation_suite_name in [es for es in context.list_expectation_suite_names() if 'segs_activity_daily' in es and 'constraints' in es]:
    print(current_expectation_suite_name)
    current_expectation_suite=context.get_expectation_suite(current_expectation_suite_name)    
    current_expectation_suite_asset_name=current_expectation_suite.meta.get('asset_name')
    current_expectation_suite_datasource_name=current_expectation_suite.meta.get('datasource_name')
    current_expectation_suite_version_number=current_expectation_suite.meta.get('version_number')

    gx_asset=gx_datasource.get_asset(current_expectation_suite_asset_name)
    gx_splitter=gx_asset.splitter
    if gx_splitter is not None:
        DATE_PARTITION_COLUMN=gx_splitter.column_name
        br_options={DATE_PARTITION_COLUMN: '2023-04-01'}
    else:
        br_options={}
    gx_br = gx_asset.build_batch_request(br_options)
    gx_batches = gx_datasource.get_batch_list_from_batch_request(gx_br)
    gx_validator = context.get_validator_using_batch_list(current_expectation_suite, gx_batches)

    gx_validator.expect_column_values_to_be_unique('seg_id')
    gx_validator.expect_column_values_to_not_be_null('seg_id')

    hours_cols = [hours_col for hours_col in gx_validator.columns() if 'hour' in hours_col and not '.' in hours_col]
    for current_hours_col in hours_cols:
        gx_validator.expect_column_values_to_be_between(
            column=current_hours_col, 
            min_value=0, 
            max_value=48, 
            strict_max=True
        )
        if current_hours_col != 'hours':
            gx_validator.expect_column_pair_values_a_to_be_greater_than_b(
                'hours', current_hours_col, or_equal=True
            )

    gx_validator.save_expectation_suite(discard_failed_expectations=False)

DEBUG:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.__fields_set__ assets added
INFO:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.dict() - substituting config values


gfw-google-827.constraints.segs_activity_daily.2-5


DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/a2cbac34-dc0d-4c4f-8fc7-09afbe00a7c0?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/jobs/a2cbac34-dc0d-4c4f-8fc7-09afbe00a7c0?location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.__fields_set__ assets added
INFO:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.dict() - substituting config values
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:44

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/datasets/tech_great_expectations_temp_ttl_7d/tables/gx_temp_a75c9e4d?prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:Attempting query SELECT count(*) AS "table.row_count" 
FROM gx_temp_a75c9e4d
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/d1aeec80-f0d4-412c-b510-9d794d22047a?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:SqlAlchemyExecutionEngine computed 1 metrics on domain_id 193ba3158932159285fb120d0c39e1da
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing

...

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/datasets/tech_great_expectations_temp_ttl_7d/tables/gx_temp_f0d40464?prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:Attempting query SELECT count(*) AS "table.row_count" 
FROM gx_temp_f0d40464
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/5611f7d1-d9fe-433e-930f-233d0200158f?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:SqlAlchemyExecutionEngine computed 1 metrics on domain_id dbc862eb3bfc23e66bc15103eab2a7c2
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing

Calculating Metrics:   0%|          | 0/11 [00:00<?, ?it/s]

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/datasets/tech_great_expectations_temp_ttl_7d/tables/gx_temp_f0d40464?prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:Attempting query SELECT count(*) AS "table.row_count" 
FROM gx_temp_f0d40464
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/9a96bc71-cc05-40d7-a960-37fc729134b3?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:SqlAlchemyExecutionEngine computed 1 metrics on domain_id dbc862eb3bfc23e66bc15103eab2a7c2
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/datasets/tech_great_expectations_temp_ttl_7d/tables/gx_temp_f0d40464?prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:Attempting query SELECT count(*) AS "table.row_count" 
FROM gx_temp_f0d40464
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/419fedcb-0c71-4fc7-80af-83f850869d39?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:SqlAlchemyExecutionEngine computed 1 metrics on domain_id dbc862eb3bfc23e66bc15103eab2a7c2
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing